In [3]:
# ============================================================
# RANDOM FOREST MODEL
# RandomizedSearchCV → GridSearchCV → OOF Evaluation
# LendingClub Interest Rate Prediction
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.base import clone

In [4]:
# ============================================================
# 1. LOAD DATA
# ============================================================

train = pd.read_csv("LC_train.csv", na_values=["NA"])
test = pd.read_csv("LC_test.csv", na_values=["NA"])

TARGET = "int_rate"
ID_COL = "ID"

print("Train shape:", train.shape)
print("Test shape:", test.shape)

Train shape: (100000, 39)
Test shape: (10000, 39)


In [5]:
# ============================================================
# 2. REMOVE LEAKAGE COLUMNS
# ============================================================

leakage_cols = [
    "loan_status",
    "grade",
    "sub_grade",
    "installment"
]

train = train.drop(columns=[col for col in leakage_cols if col in train.columns])
test = test.drop(columns=[col for col in leakage_cols if col in test.columns])

print("Remaining train columns:", train.shape[1])
print("Remaining test columns:", test.shape[1])

Remaining train columns: 38
Remaining test columns: 38


In [6]:
# ============================================================
# 3. FEATURE ENGINEERING
# ============================================================

def feature_engineering(df):
    df = df.copy()

    if "fico_range_low" in df.columns and "fico_range_high" in df.columns:
        df["fico_mid"] = (df["fico_range_low"] + df["fico_range_high"]) / 2
        df = df.drop(columns=["fico_range_low", "fico_range_high"])

    if "term" in df.columns:
        df["term_months"] = (
            df["term"]
            .astype(str)
            .str.extract(r"(\d+)", expand=False)
            .astype(float)
        )

    if "emp_length" in df.columns:
        emp_map = {
            "< 1 year": 0,
            "1 year": 1,
            "2 years": 2,
            "3 years": 3,
            "4 years": 4,
            "5 years": 5,
            "6 years": 6,
            "7 years": 7,
            "8 years": 8,
            "9 years": 9,
            "10+ years": 10
        }
        df["emp_length_num"] = df["emp_length"].map(emp_map)

    if "loan_amnt" in df.columns and "annual_inc" in df.columns:
        df["loan_to_income"] = df["loan_amnt"] / (df["annual_inc"] + 1)

    if "dti" in df.columns and "annual_inc" in df.columns:
        df["dti_income_interaction"] = df["dti"] * np.log1p(df["annual_inc"])

    if "fico_mid" in df.columns and "revol_util" in df.columns:
        df["fico_revol_interaction"] = df["fico_mid"] * df["revol_util"]

    log_cols = [
        "annual_inc",
        "loan_amnt",
        "revol_bal",
        "tot_cur_bal",
        "total_bal_ex_mort",
        "tot_coll_amt"
    ]

    for col in log_cols:
        if col in df.columns:
            df[col + "_log"] = np.log1p(df[col].clip(lower=0))

    missing_cols = [
        "mths_since_last_record",
        "mths_since_recent_inq",
        "mths_since_rcnt_il",
        "mths_since_recent_bc"
    ]

    for col in missing_cols:
        if col in df.columns:
            df[col + "_missing"] = df[col].isna().astype(int)

    return df


train_fe = feature_engineering(train)
test_fe = feature_engineering(test)

print("Feature engineered train shape:", train_fe.shape)
print("Feature engineered test shape:", test_fe.shape)

Feature engineered train shape: (100000, 52)
Feature engineered test shape: (10000, 52)


In [7]:
# ============================================================
# 4. SPLIT FEATURES AND TARGET
# ============================================================

X = train_fe.drop(columns=[TARGET])
y = train_fe[TARGET]

test_ids = test_fe[ID_COL]
X_test = test_fe.drop(columns=[ID_COL])

X_test = X_test.reindex(columns=X.columns, fill_value=np.nan)

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("X_test:", X_test.shape)

X_train: (80000, 51)
X_valid: (20000, 51)
X_test: (10000, 51)


In [8]:
# ============================================================
# 5. PREPROCESSING PIPELINE
# ============================================================

numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [9]:
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV

rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        random_state=42,
        n_jobs=-1
    ))
])

param_grid = {
    "model__n_estimators": [50, 80, 120],
    "model__max_depth": [15, 20],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2]
}

random_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_grid,
    n_iter=4,        # shorter
    scoring="neg_root_mean_squared_error",
    cv=2,            # shorter
    random_state=42,
    n_jobs=-1,
    verbose=1
)

random_search.fit(X_train, y_train)

print("Best Random Search Params:")
print(random_search.best_params_)

Fitting 2 folds for each of 4 candidates, totalling 8 fits
Best Random Search Params:
{'model__n_estimators': 120, 'model__min_samples_split': 2, 'model__min_samples_leaf': 2, 'model__max_depth': 15}


In [10]:
grid_param_grid = {
    "model__n_estimators": [
        random_search.best_params_["model__n_estimators"]
    ],
    "model__max_depth": [
        random_search.best_params_["model__max_depth"]
    ],
    "model__min_samples_split": [
        random_search.best_params_["model__min_samples_split"]
    ],
    "model__min_samples_leaf": [
        random_search.best_params_["model__min_samples_leaf"]
    ]
}

grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=grid_param_grid,
    scoring="neg_root_mean_squared_error",
    cv=2,            # shorter
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

best_rf = grid_search.best_estimator_

print("Best Grid Search Params:")
print(grid_search.best_params_)

Fitting 2 folds for each of 1 candidates, totalling 2 fits
Best Grid Search Params:
{'model__max_depth': 15, 'model__min_samples_leaf': 2, 'model__min_samples_split': 2, 'model__n_estimators': 120}


In [11]:
# ============================================================
# 8. VALIDATION SET PERFORMANCE
# ============================================================

train_pred = best_rf.predict(X_train)
valid_pred = best_rf.predict(X_valid)

train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
valid_rmse = np.sqrt(mean_squared_error(y_valid, valid_pred))
valid_mae = mean_absolute_error(y_valid, valid_pred)
valid_r2 = r2_score(y_valid, valid_pred)

print("Final Tuned Random Forest")
print("Train RMSE:", train_rmse)
print("Validation RMSE:", valid_rmse)
print("Validation MAE:", valid_mae)
print("Validation R2:", valid_r2)

Final Tuned Random Forest
Train RMSE: 2.920071593031399
Validation RMSE: 4.018550825456905
Validation MAE: 3.0073942515367658
Validation R2: 0.4103673233910319


In [12]:
# ============================================================
# 9. OOF PREDICTIONS
# ============================================================

kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X_train))

for fold, (train_idx, valid_idx) in enumerate(kf.split(X_train)):
    print(f"Training OOF fold {fold + 1}")

    X_tr = X_train.iloc[train_idx]
    X_val = X_train.iloc[valid_idx]
    y_tr = y_train.iloc[train_idx]

    fold_model = clone(best_rf)
    fold_model.fit(X_tr, y_tr)

    oof_preds[valid_idx] = fold_model.predict(X_val)

oof_rmse = np.sqrt(mean_squared_error(y_train, oof_preds))
oof_mae = mean_absolute_error(y_train, oof_preds)
oof_r2 = r2_score(y_train, oof_preds)

print("OOF Random Forest Performance")
print("OOF RMSE:", oof_rmse)
print("OOF MAE:", oof_mae)
print("OOF R2:", oof_r2)

Training OOF fold 1
Training OOF fold 2
Training OOF fold 3
Training OOF fold 4
Training OOF fold 5
OOF Random Forest Performance
OOF RMSE: 4.055389106661768
OOF MAE: 3.0481905545195596
OOF R2: 0.4058207651801835


In [13]:
# ============================================================
# 10. RESULTS TABLE
# ============================================================

results = pd.DataFrame({
    "Metric": [
        "Randomized Search CV RMSE",
        "Grid Search CV RMSE",
        "Validation RMSE",
        "Validation MAE",
        "Validation R2",
        "OOF RMSE",
        "OOF MAE",
        "OOF R2"
    ],
    "Value": [
        -random_search.best_score_,
        -grid_search.best_score_,
        valid_rmse,
        valid_mae,
        valid_r2,
        oof_rmse,
        oof_mae,
        oof_r2
    ]
})

results

,Metric,Value
0,Randomized Search CV RMSE,4.072877
1,Grid Search CV RMSE,4.072877
2,Validation RMSE,4.018551
3,Validation MAE,3.007394
4,Validation R2,0.410367
5,OOF RMSE,4.055389
6,OOF MAE,3.048191
7,OOF R2,0.405821
